# Random K-fold · 03 — Test evaluation (run once)

| § | Step |
|---|---|
| 1 | Load the three saved final models and the test features (no labels yet) |
| 2 | Encode with the manifest maps, apply the saved pooling, predict |
| 3 | Read `test_labels.csv` — **the only place it is read** — and compute metrics |
| 4 | Row and supplier-cluster bootstraps (1,000 resamples, seed 42, shared by all models) |
| 5 | Coverage: unseen levels, volume features above the train maximum; daily table |
| 6 | CV estimate vs test AUC; save |

Everything upstream is fixed: features, tree counts and models were saved before this notebook. Nothing here feeds back into any decision.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

# ========================================= Settings =========================================
STRATEGY = 'kfold_cv'
SEED = 42

RARE_CODE = -1
UNSEEN_CODE = -2
BOOTSTRAP_B = 1000
# ============================================================================================

In [2]:
DATA = Path('../Data/Processed_kfold_cv')
TARGET, ID_COL, DATE_COL = 'loan_default', 'UniqueID', 'disb_date'

manifest = json.loads((DATA / 'manifest.json').read_text())
NUMERIC = manifest['numeric_columns']
CATEGORICAL = manifest['categorical_columns']


def encode_categoricals(df: pd.DataFrame) -> None:
    """In place: raw values -> integer codes from the manifest maps (index in the list = code); unseen -> -2."""
    for col in CATEGORICAL:
        lookup = {v: i for i, v in enumerate(manifest['category_maps'][col])}
        df[col] = df[col].astype(str).map(lookup).fillna(UNSEEN_CODE).astype('int32')


def apply_pool(col: pd.Series, levels: np.ndarray) -> pd.Series:
    """Map levels outside `levels` (including UNSEEN_CODE) to RARE_CODE, as a `category` Series."""
    pooled = col.where(col.isin(levels), RARE_CODE)
    return pd.Series(pd.Categorical(pooled, categories=np.append(levels, RARE_CODE)), index=col.index)

## 1. Load models and test features

In [3]:
import joblib

cv_results = json.loads((DATA / 'cv_results.json').read_text())
MODELS = DATA / 'models'
final_spec = json.loads((MODELS / 'final_models.json').read_text())
pooling = json.loads((MODELS / 'pooling_levels.json').read_text())
chosen_family = cv_results['choice']['chosen_family']

lgbm_model = joblib.load(MODELS / 'lightgbm_final.joblib')
ctrl_model = joblib.load(MODELS / 'control_final.joblib')
cat_model = CatBoostClassifier()
cat_model.load_model(str(MODELS / 'catboost_final.cbm'))

test = pd.read_csv(DATA / 'test_features.csv', parse_dates=[DATE_COL])
test_raw_supplier = test['supplier_id'].to_numpy().copy()
test_ids, test_dates = test[ID_COL].to_numpy(), test[DATE_COL]

# coverage figures need the raw encoding step, so compute them as the frame is encoded
encode_categoricals(test)
unseen_share = {c: float((test[c] == UNSEEN_CODE).mean()) for c in CATEGORICAL}
VOL_COLS = ['supplier_id_loans_7d', 'branch_id_loans_7d', 'Employee_code_ID_loans_7d']
train_max = pd.read_csv(DATA / 'train_features.csv', usecols=VOL_COLS).max()
above_train_max = {c: float((test[c] > train_max[c]).mean()) for c in VOL_COLS}
for col in NUMERIC:
    if test[col].dtype == 'float64':
        test[col] = test[col].astype('float32')
print(f'test: {test.shape}   unseen-level share (%):')
print((pd.Series(unseen_share) * 100).round(2).to_string())

test: (50653, 50)   unseen-level share (%):
branch_id                        0.00
supplier_id                      0.80
manufacturer_id                  0.00
Current_pincode_ID               0.85
State_ID                         0.00
Employee_code_ID                 0.99
Employment.Type                  0.00
PERFORM_CNS.SCORE.DESCRIPTION    0.00


## 2. Predict

LightGBM models use the saved pooling levels (unseen and rare levels go to `-1`); CatBoost receives the integer codes directly.

In [4]:
def lgbm_test_frame(feats, levels):
    f = test[feats].copy()
    for c in feats:
        if c in CATEGORICAL:
            f[c] = apply_pool(f[c], np.asarray(levels[c]))
    return f


def cat_test_pool(feats):
    return Pool(test[feats], cat_features=[i for i, c in enumerate(feats) if c in CATEGORICAL])


p_lightgbm = lgbm_model.predict_proba(lgbm_test_frame(final_spec['lightgbm']['features'], pooling['lightgbm']))[:, 1]
p_control = ctrl_model.predict_proba(lgbm_test_frame(final_spec['control']['features'], pooling['control']))[:, 1]
p_catboost = cat_model.predict_proba(cat_test_pool(final_spec['catboost']['features']))[:, 1]
PRED = {'lightgbm': p_lightgbm, 'catboost': p_catboost, 'control': p_control}
print({k: float(v.mean()) for k, v in PRED.items()})

{'lightgbm': 0.20607532016390828, 'catboost': 0.20554049578188957, 'control': 0.20607532016390828}


## 3. Labels and metrics

Gini = 2·AUC − 1; KS = max(TPR − FPR); PR-AUC = average precision.

In [5]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_curve

labels = pd.read_csv(DATA / 'test_labels.csv')
assert np.array_equal(labels[ID_COL].to_numpy(), test_ids), 'label order differs from the test features'
y_test = labels[TARGET].to_numpy()
n_test = len(y_test)


def gini(t, p):
    return 2 * roc_auc_score(t, p) - 1


def ks(t, p):
    fpr, tpr, _ = roc_curve(t, p)
    return float(np.max(tpr - fpr))


def metrics(t, p):
    return {'auc': roc_auc_score(t, p), 'gini': gini(t, p), 'ks': ks(t, p),
            'pr_auc': average_precision_score(t, p), 'brier': brier_score_loss(t, p),
            'mean_predicted': float(np.mean(p)), 'actual_rate': float(np.mean(t))}


test_metrics = {k: metrics(y_test, p) for k, p in PRED.items()}
print(pd.DataFrame(test_metrics).T.round(4).to_string())

             auc    gini      ks  pr_auc   brier  mean_predicted  actual_rate
lightgbm  0.6470  0.2941  0.2133  0.3719  0.1835          0.2061       0.2559
catboost  0.6605  0.3209  0.2320  0.3888  0.1814          0.2055       0.2559
control   0.6470  0.2941  0.2133  0.3719  0.1835          0.2061       0.2559


## 4. Bootstraps (1,000 resamples each, the same resamples for every model)

- **Row bootstrap:** `rng = default_rng(42)`; each resample is `rng.integers(0, n, n)`.
- **Supplier-cluster bootstrap:** `rng = default_rng(42)`; each resample draws `len(S)` suppliers with replacement from `S` (the sorted unique test `supplier_id` values) and takes all their test rows.

95% percentile intervals of each model's AUC, and paired intervals for (chosen family − other family) and (chosen family − control).

In [6]:
def row_resamples(n, b=BOOTSTRAP_B, seed=SEED):
    rng = np.random.default_rng(seed)
    for _ in range(b):
        yield rng.integers(0, n, n)


S, inverse = np.unique(test_raw_supplier, return_inverse=True)
order = np.argsort(inverse, kind='stable')
starts = np.searchsorted(inverse[order], np.arange(len(S)), side='left')
ends = np.searchsorted(inverse[order], np.arange(len(S)), side='right')


def cluster_resamples(b=BOOTSTRAP_B, seed=SEED):
    rng = np.random.default_rng(seed)
    for _ in range(b):
        picks = rng.integers(0, len(S), len(S))
        yield np.concatenate([order[starts[j]:ends[j]] for j in picks])


def boot_aucs(resamples):
    out = {k: np.empty(BOOTSTRAP_B) for k in PRED}
    for b, idx in enumerate(resamples):
        for k, p in PRED.items():
            out[k][b] = roc_auc_score(y_test[idx], p[idx])
    return out


boot_row = boot_aucs(row_resamples(n_test))
boot_sup = boot_aucs(cluster_resamples())
print(f'{len(S)} test suppliers')

other_family = 'lightgbm' if chosen_family == 'catboost' else 'catboost'


def ci(a):
    lo, hi = np.percentile(a, [2.5, 97.5])
    return [float(lo), float(hi)]


auc_ci = {k: {'row': ci(boot_row[k]), 'supplier': ci(boot_sup[k])} for k in PRED}
pair_defs = {f'{chosen_family}_minus_{other_family}': (chosen_family, other_family),
             f'{chosen_family}_minus_control': (chosen_family, 'control')}
paired = {name: {'point': float(test_metrics[a]['auc'] - test_metrics[b]['auc']),
                 'row': ci(boot_row[a] - boot_row[b]), 'supplier': ci(boot_sup[a] - boot_sup[b])}
          for name, (a, b) in pair_defs.items()}
print(json.dumps(paired, indent=2))

2642 test suppliers
{
  "catboost_minus_lightgbm": {
    "point": 0.013430102415032286,
    "row": [
      0.00983083597442003,
      0.016443082714086458
    ],
    "supplier": [
      0.009084683231858601,
      0.017708719951841256
    ]
  },
  "catboost_minus_control": {
    "point": 0.013430102415032286,
    "row": [
      0.00983083597442003,
      0.016443082714086458
    ],
    "supplier": [
      0.009084683231858601,
      0.017708719951841256
    ]
  }
}


## 5. Daily table and coverage

In [7]:
daily = (pd.DataFrame({'day': test_dates.to_numpy(), 'actual': y_test, 'pred': PRED[chosen_family]})
         .groupby('day').agg(loans=('actual', 'size'), actual_rate=('actual', 'mean'), mean_predicted=('pred', 'mean')))
print(daily.round(4).to_string())
print('\nshare of test rows above the train maximum:', {k: round(v, 4) for k, v in above_train_max.items()})

            loans  actual_rate  mean_predicted
day                                           
2018-10-23   6440       0.2402          0.1929
2018-10-24   6701       0.2321          0.1963
2018-10-25   5944       0.2465          0.1985
2018-10-26   6215       0.2599          0.1986
2018-10-27   4273       0.2467          0.1997
2018-10-28   2028       0.2377          0.1939
2018-10-29   4389       0.2563          0.2061
2018-10-30   5837       0.2573          0.2143
2018-10-31   8826       0.2965          0.2310

share of test rows above the train maximum: {'supplier_id_loans_7d': 0.0249, 'branch_id_loans_7d': 0.1033, 'Employee_code_ID_loans_7d': 0.0004}


## 6. CV estimate vs test AUC, and save

How far this strategy's own cross-validated AUC (notebook 02) sits from the AUC achieved on the test window.

In [8]:
cv_vs_test = {}
for name, key, model_key in [('chosen', chosen_family, chosen_family), ('control', 'control', 'control')]:
    est = cv_results[key]['mean_auc']
    real = test_metrics[model_key]['auc']
    cv_vs_test[name] = {'estimate': est, 'realised': real, 'gap': est - real}
print(cv_vs_test)

pd.DataFrame({ID_COL: test_ids, DATE_COL: test_dates.to_numpy(), 'supplier_id': test_raw_supplier,
              TARGET: y_test, 'p_lightgbm': p_lightgbm, 'p_catboost': p_catboost,
              'p_control': p_control}).to_csv(DATA / 'test_predictions.csv', index=False)

(DATA / 'test_results.json').write_text(json.dumps({
    'strategy': STRATEGY, 'n_test': int(n_test), 'n_test_suppliers': int(len(S)),
    'chosen_family': chosen_family, 'other_family': other_family,
    'metrics': test_metrics, 'auc_ci95': auc_ci, 'paired': paired,
    'bootstrap': {'resamples': BOOTSTRAP_B, 'seed': SEED},
    'daily': [{'day': str(d.date()), 'loans': int(r['loans']), 'actual_rate': float(r['actual_rate']),
               'mean_predicted': float(r['mean_predicted'])} for d, r in daily.iterrows()],
    'unseen_level_share': unseen_share, 'volume_above_train_max_share': above_train_max,
    'cv_estimate_vs_test': cv_vs_test,
}, indent=2))
print('Wrote', DATA / 'test_predictions.csv', DATA / 'test_results.json')

{'chosen': {'estimate': 0.6788964849232257, 'realised': 0.6604565897885792, 'gap': 0.0184398951346465}, 'control': {'estimate': 0.6608362238493057, 'realised': 0.647026487373547, 'gap': 0.013809736475758783}}
Wrote ..\Data\Processed_kfold_cv\test_predictions.csv ..\Data\Processed_kfold_cv\test_results.json
